# Testing Context Extraction with LLM

In [266]:
# imports
from urllib import request
import json


In [ ]:
# API
url = "http://localhost:11434/api/generate"

 This is a simple recursive function that takes an integer `n` as input, and decrements it by one until it reaches zero. The loop will terminate when `n` reaches zero, because the condition `n > 0` becomes false at that point.

Therefore, this function terminates for any non-negative value of `n`.


## Test Examples

In [249]:
fileCode1 = """
# Globale Variablen
a = 10
b = 20
c = 30
d = 5

# Funktionen
def func1():
    return a + helper1()

def helper1():
    return b + helper2()

def helper2():
    return 42

def dummy1():
    test = 5;
    return test
"""

functionToAnalyze1 = "func1"

fileCode2 = """
# Globale Variablen
limit = 10
offset = 3
factor = 2
unused = 999

# Funktionen
def func2(n):
    if n > limit:
        return helperA(n) + offset
    else:
        return helperB(n)

def helperA(x):
    return x * factor

def helperB(x):
    return x + factor

def dummy2():
    return unused
"""

functionToAnalyze2 = "func2"

fileCode3 = """
# Globale Variablen
threshold = 100
step = 3
start = 1

# Funktionen
def func3():
    value = start
    while value < threshold:
        value = helper3(value)
    return value

def helper3(x):
    return x + step

def dummy3():
    return threshold
"""

functionToAnalyze3 = "func3"

fileCode4 = """
# Globale Variablen
base = 0
decrement = 1

# Funktionen
def func4(n):
    if is_base(n):
        return n
    return func4(n - decrement)

def is_base(x):
    return x <= base

def helper_unused():
    return decrement
"""

functionToAnalyze4 = "func4"




In [ ]:
def find_called_functions_names(full_code, target_fn):
    prompt = f"""You are a static program analysis assistant. 

    Full source code:
    {full_code}

    Target function:
    {target_fn}

    Task:
    - Create a list containing the 'Target Function' and all functions that are directly or indirectly called by the 'Target Function'. 
    - Do not include functions that are not directly or indirectly called by the 'Target Function'.

    Rules:
    - Output ONLY a JSON array of function names, e.g. ["helper1", "helper2"]
    - No explanations, no extra text.
    """

    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted


def find_all_global_variables(full_code):
    prompt = f"""You are a static program analysis assistant.

    Full source code:
    {full_code}

    Task:
    - Identify all global variables that are defined in the 'Full source code'.

    Rules:
    - Output ONLY a JSON array of variable names, e.g. ["a", "b"]
    - No explanations, no extra text.
    """

    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted

def summarize_called_functions(full_code, called_functions):
    prompt = f"""You are a static program analysis assistant.

    Full source code:
    {full_code}

    Relevant Functions:
    {called_functions}

    Task:
    - Extract only the definitions of the functions listed in 'Relevant Functions'.
    - Do NOT include any other functions or global variables.

    Rules:
    - Output valid Python code.
    - Preserve indentation and line breaks.
    - No explanations, no comments, no extra text.
    - Do not add comments, explanations, or language identifiers.
    """

    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted

def find_all_variable_dependencies(called_functions_summary, global_variables):
    prompt = f"""You are a static program analysis assistant.

    Sourcecode:
    {called_functions_summary}

    Global Variables:
    {global_variables}

    Task:
    - Look at every function in 'Sourcecode' and identify the variables that are being used, that are also in 'Gloabal Variables'.

    Rules:
    - Output ONLY a JSON array of variable names, e.g. ["a", "b"]
    - No explanations, no extra text.
    """

    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted

def summarize_called_global_variables(full_code, called_global_variables):
    prompt = f"""You are a static program analysis assistant.

    Sourcecode:
    {full_code}

    Global Variables:
    {called_global_variables}

    Task:
    - Identify the definitions for the variables from 'Global Variables' in 'Sourcecode' and summarize them into one String.

    Rules:
    - Output valid Python code.
    - Preserve indentation and line breaks.
    - Output ONLY
    - No explanations, no extra text.
    """

    payload = {
        "model": "codellama:13b",
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "top_p": 1.0
        }
    }

    req = request.Request(url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )

    with request.urlopen(req) as response:
        raw = json.loads(response.read())["response"]

    # --- normalize newlines (robust against escaped \\n) ---
    formatted = raw.replace("\\n", "\n").strip()

    return formatted


In [272]:
fullFileCode = fileCode2
finalFunctionToAnalyse = functionToAnalyze2

called_functions = find_called_functions_names(finalFunctionToAnalyse, fullFileCode)
global_variables = find_all_global_variables(fullFileCode)
called_functions_summary = summarize_called_functions(fullFileCode, called_functions)
called_global_variables = find_all_variable_dependencies(called_functions_summary, global_variables)
called_global_variables_summary = summarize_called_global_variables(fullFileCode, called_global_variables)
result = called_global_variables_summary + "\n" + "\n" + called_functions_summary

print("\n" + "="*30)
print("✅ Called Functions")
print(called_functions)

print("\n" + "="*30)
print("✅ All Global Variables")
print(global_variables)

print("\n" + "="*30)
print("✅ Called Functions Summary")
print(called_functions_summary)

print("\n" + "="*30)
print("✅ Called Global Variables")
print(called_global_variables)

print("\n" + "="*30)
print("✅ Called Global Variables Summary")
print(called_global_variables_summary)




✅ Called Functions
["func2", "helperA", "helperB"]

✅ All Global Variables
["limit", "offset", "factor", "unused"]

✅ Called Functions Summary
# Globale Variablen
limit = 10
offset = 3
factor = 2
unused = 999

# Funktionen
def func2(n):
    if n > limit:
        return helperA(n) + offset
    else:
        return helperB(n)

def helperA(x):
    return x * factor

def helperB(x):
    return x + factor

✅ Called Global Variables
["limit", "offset", "factor"]

✅ Called Global Variables Summary
limit = 10
    offset = 3
    factor = 2


In [273]:
# Fix Code Structure
def reindent_functions(code: str) -> str:
    lines = code.splitlines()
    result = []

    indent_level = 0
    block_openers = ("if ", "for ", "while ", "try:")
    block_continuations = ("elif ", "else:", "except", "finally:")

    # Tracken, ob die vorherige Zeile ein Blockstarter war
    prev_was_block = False

    for line in lines:
        stripped = line.strip()

        if stripped == "":
            result.append("")
            continue

        # Neue Funktion → Reset
        if stripped.startswith("def ") and stripped.endswith(":"):
            indent_level = 0
            result.append(stripped)
            indent_level = 1
            prev_was_block = True
            continue

        # else / elif / except / finally → dedent first
        if stripped.startswith(block_continuations):
            indent_level -= 1
            result.append("    " * indent_level + stripped)
            indent_level += 1
            prev_was_block = True
            continue

        # if / for / while / try
        if stripped.endswith(":") and stripped.startswith(block_openers):
            result.append("    " * indent_level + stripped)
            indent_level += 1
            prev_was_block = True
            continue

        # normale Anweisung
        # Wenn vorher ein Blockstarter war, normale Anweisung einrücken
        result.append("    " * indent_level + stripped)
        prev_was_block = False

    return "\n".join(result)




fixed = reindent_functions(result)

print(fixed)


limit = 10
offset = 3
factor = 2

# Globale Variablen
limit = 10
offset = 3
factor = 2
unused = 999

# Funktionen
def func2(n):
    if n > limit:
        return helperA(n) + offset
    else:
        return helperB(n)

def helperA(x):
    return x * factor

def helperB(x):
    return x + factor
